# 🏦 Банковский счёт: `Account` и `CreditAccount`

> **Цель:** реализовать класс `Account`, моделирующий банковский счёт, с историей операций и аналитикой.  
> Затем создать наследника `CreditAccount`.

---

## 📌 1. Класс `Account`

### Инициализация

```python
Account(account_holder, balance=0)
```

| Параметр | Тип | По умолчанию | Описание |
|----------|-----|--------------|----------|
| `account_holder` | `str` | — | Имя владельца счёта |
| `balance` | `float` | `0` | Начальный баланс. Не может быть отрицательным |

### Атрибуты

| Атрибут | Тип | Описание |
|---------|-----|----------|
| `holder` | `str` | Имя владельца |
| `_balance` | `float` | Приватный текущий баланс |
| `operations_history` | `list` / `tuple` | История операций |

> ⚠️ **Важно:** каждая операция хранится как структурированная запись — словарь или кортеж.  
> Минимальный набор полей:
> - тип операции: `'deposit'` или `'withdraw'`;
> - сумма;
> - дата и время;
> - текущий баланс после операции;
> - статус: `'success'` или `'fail'`.

#### Пример записи операции

```python
{
    "type": "deposit",
    "amount": 1000.0,
    "datetime": datetime.now(),
    "balance_after": 1000.0,
    "status": "success"
}
```

---

## 🛠️ 2. Методы `Account`

### `__init__(self, account_holder, balance=0)`
Конструктор класса.

### `deposit(self, amount)`
Пополнение счёта.

- принимает сумму;
- сумма должна быть положительной;
- при успехе обновляет баланс и добавляет запись в историю.

### `withdraw(self, amount)`
Снятие средств.

- принимает сумму;
- сумма должна быть положительной;
- проверяет, достаточно ли средств;
- если средств недостаточно — операция не проходит, но попытка фиксируется в истории со статусом `'fail'`;
- при успехе обновляет баланс и добавляет запись.

### `get_balance(self)`
Возвращает текущий баланс.

### `get_history(self)`
Возвращает историю операций.

> 💡 **Подсказка:** для работы с датой и временем используйте `datetime.now()` из модуля `datetime`.

---

## 💳 3. Класс `CreditAccount(Account)`

Наследует всю функциональность `Account` и добавляет кредитную логику.

### Особенности

- При инициализации принимает дополнительный параметр `credit_limit`.
- Баланс может быть отрицательным, но не ниже `-credit_limit`.
- По запросу показывает доступные кредитные средства:  
  `доступно = текущий баланс + credit_limit`
- В историю операций добавляется информация о том, были ли использованы кредитные средства.

### Инициализация

```python
CreditAccount(account_holder, balance=0, credit_limit=0)
```

---

## ✅ Чек-лист реализации

- [ ] Создан класс `Account`.
- [ ] Реализован конструктор с проверкой баланса.
- [ ] Реализованы `deposit`, `withdraw`, `get_balance`, `get_history`.
- [ ] История операций хранит структурированные записи.
- [ ] Неудачные попытки снятия фиксируются.
- [ ] Создан класс `CreditAccount`.
- [ ] Учтён кредитный лимит.
- [ ] Добавлена информация об использовании кредитных средств.

---

## 📚 Полезные импорты

```python
from datetime import datetime
```

---

> **Совет:** для красивого вывода истории можно вернуть список словарей или преобразовать его в `pandas.DataFrame`.

# 📊 Критерии оценивания проекта

> **Максимум по базовым критериям: 8 баллов** — это оценка **«отлично»**.  
> 💡 Преподаватель может добавить **1–2 балла** и поставить **9 или 10**, если работа выполнена сверх ожиданий и выходит за рамки программы.

---

| № | Критерий | Баллы | Описание |
|:--:|:---------|:-----:|:---------|
| 1 | 🧱 **Корректность ООП** | **3** | Класс корректно инициализируется, используются инкапсуляция (приватные атрибуты, геттеры), методы `deposit` и `withdraw` работают без ошибок, логически верно обрабатывают неверные входные данные (отрицательные суммы) |
| 2 | 📜 **История операций** | **2** | История сохраняется в структурированном виде. Метод `get_history` возвращает информацию в удобном формате |
| 3 | 🕒 **Работа с датами** | **1** | В каждой операции корректно фиксируется дата и время её проведения с помощью модуля `datetime` |
| 4 | 🧬 **Наследование** | **2** | Корректно реализована функциональность класса `CreditAccount(Account)`: в реализации учтены все заданные особенности кредитного счёта |
| 5 | ✨ **Чистота кода** | **−1** за каждое нарушение | Код откомментирован, читаемый, соответствует PEP8 |

---

> **Итого:** `3 + 2 + 1 + 2 = 8` баллов — оценка **«отлично»**.  
> 🚀 Дополнительные баллы: **9–10**, если работа выходит за рамки программы.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_EVEN
from enum import StrEnum
from uuid import UUID, uuid4


ZERO_MONEY = Decimal("0.00")
MONEY_QUANTUM = Decimal("0.01")


def normalize_money(value: object) -> Decimal:
    """Проверяет и округляет денежное значение до двух знаков."""

    if not isinstance(value, Decimal):
        raise TypeError("Денежное значение должно иметь тип Decimal")

    if not value.is_finite():
        raise ValueError("Денежное значение должно быть конечным числом")

    return value.quantize(
        MONEY_QUANTUM,
        rounding=ROUND_HALF_EVEN,
    )


class OperationType(StrEnum):
    """Определяет тип банковской операции."""

    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"


class OperationStatus(StrEnum):
    """Определяет результат банковской операции."""

    SUCCESS = "success"
    FAIL = "fail"


class FailureReason(StrEnum):
    """Определяет причину отказа в операции."""

    INSUFFICIENT_FUNDS = "insufficient_funds"
    CREDIT_LIMIT_EXCEEDED = "credit_limit_exceeded"
    ACCOUNT_BLOCKED = "account_blocked"


class Currency(StrEnum):
    """Определяет валюту банковского счёта."""

    RUB = "RUB"
    USD = "USD"
    EUR = "EUR"


class AccountStatus(StrEnum):
    """Определяет состояние банковского счёта."""

    ACTIVE = "active"
    BLOCKED = "blocked"


class ActorType(StrEnum):
    """Определяет тип инициатора события."""

    ACCOUNT_HOLDER = "account_holder"
    BANK_EMPLOYEE = "bank_employee"
    SYSTEM = "system"


class AccountEventType(StrEnum):
    """Определяет тип события банковского счёта."""

    BLOCKED = "blocked"
    UNBLOCKED = "unblocked"


class BlockReason(StrEnum):
    """Определяет причину блокировки счёта."""

    USER_REQUEST = "user_request"
    FRAUD_SUSPICION = "fraud_suspicion"
    BANK_DECISION = "bank_decision"
    LEGAL_RESTRICTION = "legal_restriction"


@dataclass(frozen=True, slots=True)
class EventActor:
    """Представляет инициатора события счёта."""

    actor_type: ActorType
    actor_id: UUID | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность типа и ID инициатора."""

        if self.actor_type is ActorType.SYSTEM and self.actor_id is not None:
            raise ValueError("Системный инициатор не должен иметь actor_id")

        if self.actor_type is not ActorType.SYSTEM and self.actor_id is None:
            raise ValueError("Владелец или сотрудник должен иметь actor_id")


@dataclass(frozen=True, slots=True)
class AccountEvent:
    """Представляет неизменяемое событие счёта."""

    event_id: UUID
    account_id: UUID
    event_type: AccountEventType
    created_at: datetime
    actor: EventActor
    block_reason: BlockReason | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность статуса блокировки и причины."""

        if self.event_type is AccountEventType.BLOCKED and self.block_reason is None:
            raise ValueError("Событие блокировки должно иметь причину")

        if (
            self.event_type is AccountEventType.UNBLOCKED
            and self.block_reason is not None
        ):
            raise ValueError(
                "Событие разблокировки не может содержать причину блокировки"
            )


@dataclass(frozen=True, slots=True)
class Transaction:
    """Представляет неизменяемую банковскую операцию."""

    transaction_id: UUID
    account_id: UUID
    currency: Currency
    operation_type: OperationType
    amount: Decimal
    created_at: datetime
    balance_after: Decimal
    status: OperationStatus
    failure_reason: FailureReason | None = None
    credit_used: bool | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность статуса и причины отказа."""

        if self.status is OperationStatus.SUCCESS and self.failure_reason is not None:
            raise ValueError("Успешная операция не может иметь причину отказа")

        if self.status is OperationStatus.FAIL and self.failure_reason is None:
            raise ValueError("Неуспешная операция должна иметь причину отказа")

    @property
    def is_successful(self) -> bool:
        """Показывает, была ли операция успешной."""

        return self.status is OperationStatus.SUCCESS


class Account:
    """Моделирует банковский счёт и хранит историю операций."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        currency: Currency = Currency.RUB,
        account_id: UUID | None = None,
        account_status: AccountStatus = AccountStatus.ACTIVE,
        block_reason: BlockReason | None = None,
        holder_id: UUID | None = None,
    ) -> None:
        """Создаёт банковский счёт с указанным владельцем и начальным балансом."""

        balance = normalize_money(balance)

        if balance < ZERO_MONEY:
            raise ValueError("Начальный баланс не может быть отрицательным")

        if account_id is not None and not isinstance(account_id, UUID):
            raise TypeError("Идентификатор счёта должен иметь тип UUID")

        if block_reason is not None and not isinstance(block_reason, BlockReason):
            raise TypeError("Причина блокировки должна иметь тип BlockReason")

        if not isinstance(currency, Currency):
            raise TypeError("Валюта должна иметь тип Currency")

        if not isinstance(account_status, AccountStatus):
            raise TypeError("Статус счета должен иметь тип AccountStatus")

        if account_status is AccountStatus.BLOCKED and block_reason is None:
            raise ValueError(
                "Для заблокированного счёта необходимо указать причину блокировки"
            )

        if account_status is AccountStatus.ACTIVE and block_reason is not None:
            raise ValueError("Активный счёт не может иметь причину блокировки")

        if holder_id is not None and not isinstance(holder_id, UUID):
            raise TypeError("Идентификатор владельца должен иметь тип UUID")

        self._currency: Currency = currency
        self._account_id: UUID = uuid4() if account_id is None else account_id
        self.holder: str = account_holder
        self._balance: Decimal = balance
        self._account_status: AccountStatus = account_status
        self._block_reason: BlockReason | None = block_reason
        self._holder_id: UUID = uuid4() if holder_id is None else holder_id
        self._operations_history: list[Transaction] = []
        self._events_history: list[AccountEvent] = []

    def get_holder_id(self) -> UUID:
        """Возвращает идентификатор владельца счёта."""

        return self._holder_id

    def get_account_status(self) -> AccountStatus:
        """Возвращает статус банковского счета."""

        return self._account_status

    def get_block_reason(self) -> BlockReason | None:
        """Возвращает текущую причину блокировки счёта."""

        return self._block_reason

    def block_account(
        self, block_reason: BlockReason, actor: EventActor
    ) -> AccountEvent | None:
        """Блокирует счёт по указанной причине."""

        if not isinstance(block_reason, BlockReason):
            raise TypeError("Причина блокировки должна иметь тип BlockReason")

        if not isinstance(actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if not self._can_block(block_reason, actor):
            raise PermissionError(
                "Инициатор не имеет права блокировать счёт по этой причине"
            )

        if self.is_blocked:
            return None

        self._account_status = AccountStatus.BLOCKED
        self._block_reason = block_reason

        return self._add_account_event(
            event_type=AccountEventType.BLOCKED,
            block_reason=block_reason,
            actor_event=actor,
        )

    def unblock_account(self, actor: EventActor) -> AccountEvent | None:
        """Разблокирует счёт."""

        if not isinstance(actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if not self.is_blocked:
            return None

        if not self._can_unblock(actor):
            raise PermissionError("Инициатор не имеет права разблокировать этот счёт")

        self._account_status = AccountStatus.ACTIVE
        self._block_reason = None

        return self._add_account_event(
            event_type=AccountEventType.UNBLOCKED,
            actor_event=actor,
        )

    def _get_available_withdrawal_funds(self) -> Decimal:
        """Возвращает сумму, доступную для снятия."""

        return self._balance

    @property
    def is_blocked(self) -> bool:
        """Показывает, заблокирован ли счёт."""

        return self._account_status is AccountStatus.BLOCKED

    def get_currency(self) -> Currency:
        """Возвращает валюту счёта."""

        return self._currency

    def get_account_id(self) -> UUID:
        """Возвращает идентификатор счёта."""

        return self._account_id

    def _get_insufficient_funds_reason(self) -> FailureReason:
        """Возвращает причину отказа при нехватке средств."""

        return FailureReason.INSUFFICIENT_FUNDS

    def _get_credit_used(self, status: OperationStatus) -> bool | None:
        """Возвращает признак использования кредита."""

        return None

    def deposit(self, amount: Decimal) -> Transaction:
        """Пополняет счёт на положительную сумму."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount

        return self._add_history(
            operation_type=OperationType.DEPOSIT,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=OperationStatus.SUCCESS,
        )

    def withdraw(self, amount: Decimal) -> Transaction:
        """Снимает средства в пределах доступной суммы."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма снятия должна быть положительной")

        if self.is_blocked:
            status = OperationStatus.FAIL
            failure_reason = FailureReason.ACCOUNT_BLOCKED
        elif amount > self._get_available_withdrawal_funds():
            status = OperationStatus.FAIL
            failure_reason = self._get_insufficient_funds_reason()
        else:
            self._balance -= amount
            status = OperationStatus.SUCCESS
            failure_reason = None

        credit_used = self._get_credit_used(status)

        return self._add_history(
            operation_type=OperationType.WITHDRAW,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
            failure_reason=failure_reason,
            credit_used=credit_used,
        )

    def get_balance(self) -> Decimal:
        """Возвращает текущий баланс счёта."""

        return self._balance

    def _add_history(
        self,
        operation_type: OperationType,
        amount: Decimal,
        date_time: datetime,
        balance_after: Decimal,
        status: OperationStatus,
        failure_reason: FailureReason | None = None,
        credit_used: bool | None = None,
    ) -> Transaction:
        """Добавляет запись об операции в историю счёта."""

        transaction = Transaction(
            transaction_id=uuid4(),
            account_id=self._account_id,
            currency=self._currency,
            operation_type=operation_type,
            amount=amount,
            created_at=date_time,
            balance_after=balance_after,
            status=status,
            failure_reason=failure_reason,
            credit_used=credit_used,
        )

        self._operations_history.append(transaction)
        return transaction

    def _add_account_event(
        self,
        event_type: AccountEventType,
        actor_event: EventActor,
        block_reason: BlockReason | None = None,
    ) -> AccountEvent:
        """Добавляет событие в историю счёта."""

        event = AccountEvent(
            event_id=uuid4(),
            account_id=self._account_id,
            event_type=event_type,
            created_at=datetime.now(timezone.utc),
            actor=actor_event,
            block_reason=block_reason,
        )
        self._events_history.append(event)
        return event

    def get_history(self) -> tuple[Transaction, ...]:
        """Возвращает неизменяемую историю операций."""

        return tuple(self._operations_history)

    def get_events_history(self) -> tuple[AccountEvent, ...]:
        """Возвращает неизменяемую историю событий счёта."""

        return tuple(self._events_history)

    def _can_block(
        self,
        block_reason: BlockReason,
        actor: EventActor,
    ) -> bool:
        """Проверяет право инициатора заблокировать счёт."""

        if block_reason is BlockReason.USER_REQUEST:
            return (
                self._is_account_holder(actor)
                or actor.actor_type is ActorType.BANK_EMPLOYEE
            )

        allowed_actor_types = {
            BlockReason.FRAUD_SUSPICION: {
                ActorType.BANK_EMPLOYEE,
                ActorType.SYSTEM,
            },
            BlockReason.BANK_DECISION: {
                ActorType.BANK_EMPLOYEE,
            },
            BlockReason.LEGAL_RESTRICTION: {
                ActorType.BANK_EMPLOYEE,
                ActorType.SYSTEM,
            },
        }

        return actor.actor_type in allowed_actor_types[block_reason]

    def _can_unblock(self, actor: EventActor) -> bool:
        """Проверяет право инициатора разблокировать счёт."""

        if self._block_reason is BlockReason.USER_REQUEST:
            return (
                self._is_account_holder(actor)
                or actor.actor_type is ActorType.BANK_EMPLOYEE
            )

        return actor.actor_type is ActorType.BANK_EMPLOYEE

    def _is_account_holder(self, actor: EventActor) -> bool:
        """Проверяет, является ли инициатор владельцем этого счёта."""

        return (
            actor.actor_type is ActorType.ACCOUNT_HOLDER
            and actor.actor_id == self._holder_id
        )

    def __str__(self) -> str:
        """Возвращает строковое представление банковского счёта."""

        return (
            f"Владелец: {self.holder}. Баланс: {self._balance} {self._currency.value}"
        )


class CreditAccount(Account):
    """Моделирует банковский счёт с кредитным лимитом."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        credit_limit: Decimal = ZERO_MONEY,
        currency: Currency = Currency.RUB,
        account_id: UUID | None = None,
        account_status: AccountStatus = AccountStatus.ACTIVE,
        block_reason: BlockReason | None = None,
        holder_id: UUID | None = None,
    ) -> None:
        """Создаёт банковский счёт с балансом и кредитным лимитом."""

        super().__init__(
            account_holder=account_holder,
            balance=balance,
            account_id=account_id,
            currency=currency,
            account_status=account_status,
            block_reason=block_reason,
            holder_id=holder_id,
        )

        credit_limit = normalize_money(credit_limit)

        if credit_limit < ZERO_MONEY:
            raise ValueError("Кредитный лимит не может быть отрицательным")

        self._credit_limit: Decimal = credit_limit

    def _get_available_withdrawal_funds(self) -> Decimal:
        """Возвращает средства с учётом кредитного лимита."""

        return self.get_available_funds()

    def _get_insufficient_funds_reason(self) -> FailureReason:
        """Возвращает причину превышения кредитного лимита."""

        return FailureReason.CREDIT_LIMIT_EXCEEDED

    def _get_credit_used(self, status: OperationStatus) -> bool | None:
        """Показывает, использованы ли кредитные средства."""

        return status is OperationStatus.SUCCESS and self._balance < ZERO_MONEY

    def get_available_funds(self) -> Decimal:
        """Возвращает общую сумму доступных собственных и кредитных средств."""

        return self._balance + self._credit_limit

    def get_credit_limit(self) -> Decimal:
        """Возвращает кредитный лимит."""

        return self._credit_limit

    def __str__(self) -> str:
        """Возвращает строковое представление кредитного счёта."""

        available_funds = self.get_available_funds()

        return (
            f"Владелец: {self.holder}. "
            f"Баланс: {self._balance} {self._currency.value}. "
            f"Кредитный лимит: {self._credit_limit} {self._currency.value}. "
            f"Доступно средств: {available_funds} {self._currency.value}."
        )

In [11]:
from dataclasses import FrozenInstanceError
from typing import Callable


def assert_raises(
    exception_type: type[BaseException],
    action: Callable[[], object],
) -> BaseException:
    """Проверяет, что вызов завершается указанным исключением."""

    try:
        action()
    except exception_type as error:
        return error
    except Exception as error:
        raise AssertionError(
            f"Ожидалось исключение {exception_type.__name__}, "
            f"получено {type(error).__name__}"
        ) from error

    raise AssertionError(
        f"Ожидалось исключение {exception_type.__name__}"
    )


def test_normalize_money() -> None:
    """Проверяет нормализацию денежных значений."""

    assert normalize_money(
        Decimal("10.125")
    ) == Decimal("10.12")

    assert normalize_money(
        Decimal("10.135")
    ) == Decimal("10.14")

    assert_raises(
        TypeError,
        lambda: normalize_money(10.12),
    )

    assert_raises(
        ValueError,
        lambda: normalize_money(Decimal("Infinity")),
    )


def test_account_creation() -> None:
    """Проверяет создание обычного банковского счёта."""

    # Arrange
    holder_id = uuid4()
    account_id = uuid4()

    # Act
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.126"),
        account_id=account_id,
        holder_id=holder_id,
    )

    # Assert
    assert account.get_balance() == Decimal("100.13")
    assert account.get_account_id() == account_id
    assert account.get_holder_id() == holder_id
    assert account.get_currency() is Currency.RUB
    assert account.get_account_status() is AccountStatus.ACTIVE
    assert account.get_block_reason() is None
    assert account.get_history() == ()
    assert account.get_events_history() == ()


def test_account_constructor_validation() -> None:
    """Проверяет валидацию параметров банковского счёта."""

    assert_raises(
        ValueError,
        lambda: Account(
            account_holder="Илья",
            balance=Decimal("-1.00"),
        ),
    )

    assert_raises(
        TypeError,
        lambda: Account(
            account_holder="Илья",
            balance=Decimal("0.00"),
            holder_id="не UUID",  # type: ignore[arg-type]
        ),
    )

    assert_raises(
        ValueError,
        lambda: Account(
            account_holder="Илья",
            account_status=AccountStatus.BLOCKED,
        ),
    )

    assert_raises(
        ValueError,
        lambda: Account(
            account_holder="Илья",
            block_reason=BlockReason.BANK_DECISION,
        ),
    )


def test_deposit() -> None:
    """Проверяет успешное пополнение счёта."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act
    transaction = account.deposit(Decimal("25.126"))

    # Assert
    assert transaction.operation_type is OperationType.DEPOSIT
    assert transaction.status is OperationStatus.SUCCESS
    assert transaction.amount == Decimal("25.13")
    assert transaction.balance_after == Decimal("125.13")
    assert transaction.account_id == account.get_account_id()
    assert transaction.currency is Currency.RUB
    assert transaction.failure_reason is None
    assert transaction.is_successful

    assert account.get_balance() == Decimal("125.13")
    assert account.get_history() == (transaction,)

    assert transaction.created_at.tzinfo is not None
    assert transaction.created_at.utcoffset() is not None


def test_invalid_deposit() -> None:
    """Проверяет запрет нулевого и отрицательного пополнения."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act and Assert
    assert_raises(
        ValueError,
        lambda: account.deposit(Decimal("0.00")),
    )

    assert_raises(
        ValueError,
        lambda: account.deposit(Decimal("-1.00")),
    )

    assert account.get_balance() == Decimal("100.00")
    assert account.get_history() == ()


def test_successful_withdrawal() -> None:
    """Проверяет успешное снятие денег с обычного счёта."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act
    transaction = account.withdraw(Decimal("40.00"))

    # Assert
    assert transaction.operation_type is OperationType.WITHDRAW
    assert transaction.status is OperationStatus.SUCCESS
    assert transaction.failure_reason is None
    assert transaction.credit_used is None
    assert transaction.balance_after == Decimal("60.00")

    assert account.get_balance() == Decimal("60.00")


def test_insufficient_funds() -> None:
    """Проверяет отказ при недостатке денег на обычном счёте."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Act
    transaction = account.withdraw(Decimal("101.00"))

    # Assert
    assert transaction.status is OperationStatus.FAIL

    assert (
        transaction.failure_reason
        is FailureReason.INSUFFICIENT_FUNDS
    )

    assert not transaction.is_successful
    assert transaction.balance_after == Decimal("100.00")
    assert account.get_balance() == Decimal("100.00")

    assert account.get_history() == (transaction,)


def test_credit_account() -> None:
    """Проверяет снятие денег с кредитного счёта."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=Decimal("100.00"),
        credit_limit=Decimal("200.00"),
    )

    # Act
    successful_transaction = account.withdraw(
        Decimal("250.00")
    )

    # Assert
    assert (
        successful_transaction.status
        is OperationStatus.SUCCESS
    )
    assert successful_transaction.credit_used is True
    assert successful_transaction.failure_reason is None

    assert account.get_balance() == Decimal("-150.00")
    assert account.get_credit_limit() == Decimal("200.00")
    assert account.get_available_funds() == Decimal("50.00")

    # Act
    failed_transaction = account.withdraw(
        Decimal("50.01")
    )

    # Assert
    assert (
        failed_transaction.status
        is OperationStatus.FAIL
    )

    assert (
        failed_transaction.failure_reason
        is FailureReason.CREDIT_LIMIT_EXCEEDED
    )

    assert failed_transaction.credit_used is False
    assert account.get_balance() == Decimal("-150.00")


def test_event_actor_validation() -> None:
    """Проверяет создание инициаторов событий."""

    system = EventActor(
        actor_type=ActorType.SYSTEM,
    )

    assert system.actor_id is None

    assert_raises(
        ValueError,
        lambda: EventActor(
            actor_type=ActorType.SYSTEM,
            actor_id=uuid4(),
        ),
    )

    assert_raises(
        ValueError,
        lambda: EventActor(
            actor_type=ActorType.ACCOUNT_HOLDER,
        ),
    )


def test_holder_permissions() -> None:
    """Проверяет права владельца конкретного счёта."""

    # Arrange
    holder_id = uuid4()

    owner = EventActor(
        actor_type=ActorType.ACCOUNT_HOLDER,
        actor_id=holder_id,
    )

    stranger = EventActor(
        actor_type=ActorType.ACCOUNT_HOLDER,
        actor_id=uuid4(),
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
        holder_id=holder_id,
    )

    # Другой клиент не может заблокировать счёт
    assert_raises(
        PermissionError,
        lambda: account.block_account(
            block_reason=BlockReason.USER_REQUEST,
            actor=stranger,
        ),
    )

    assert not account.is_blocked
    assert account.get_block_reason() is None
    assert account.get_events_history() == ()

    # Настоящий владелец может заблокировать счёт
    blocked_event = account.block_account(
        block_reason=BlockReason.USER_REQUEST,
        actor=owner,
    )

    assert blocked_event is not None
    assert account.is_blocked

    assert (
        account.get_block_reason()
        is BlockReason.USER_REQUEST
    )

    assert blocked_event.actor == owner

    events_before = account.get_events_history()

    # Другой клиент не может разблокировать счёт
    assert_raises(
        PermissionError,
        lambda: account.unblock_account(stranger),
    )

    assert account.is_blocked
    assert account.get_events_history() == events_before

    # Настоящий владелец может разблокировать счёт
    unblocked_event = account.unblock_account(owner)

    assert unblocked_event is not None
    assert not account.is_blocked
    assert account.get_block_reason() is None

    assert account.get_events_history() == (
        blocked_event,
        unblocked_event,
    )


def test_employee_and_system_permissions() -> None:
    """Проверяет полномочия сотрудника и системы."""

    # Arrange
    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    system = EventActor(
        actor_type=ActorType.SYSTEM,
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    # Система блокирует подозрительный счёт
    blocked_event = account.block_account(
        block_reason=BlockReason.FRAUD_SUSPICION,
        actor=system,
    )

    assert blocked_event is not None
    assert account.is_blocked

    # Система не может снять блокировку
    assert_raises(
        PermissionError,
        lambda: account.unblock_account(system),
    )

    assert account.is_blocked

    # Сотрудник снимает блокировку
    unblocked_event = account.unblock_account(employee)

    assert unblocked_event is not None
    assert not account.is_blocked
    assert account.get_block_reason() is None


def test_blocked_account_withdrawal() -> None:
    """Проверяет запрет снятия с заблокированного счёта."""

    # Arrange
    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    account.block_account(
        block_reason=BlockReason.BANK_DECISION,
        actor=employee,
    )

    # Act
    transaction = account.withdraw(Decimal("10.00"))

    # Assert
    assert transaction.status is OperationStatus.FAIL

    assert (
        transaction.failure_reason
        is FailureReason.ACCOUNT_BLOCKED
    )

    assert transaction.balance_after == Decimal("100.00")
    assert account.get_balance() == Decimal("100.00")


def test_history_is_protected() -> None:
    """Проверяет защиту истории и банковской операции."""

    # Arrange
    account = Account(
        account_holder="Илья",
        balance=Decimal("100.00"),
    )

    transaction = account.deposit(Decimal("10.00"))

    # Act
    history = account.get_history()

    # Assert
    assert isinstance(history, tuple)
    assert history == (transaction,)

    assert_raises(
        AttributeError,
        lambda: history.append(transaction),  # type: ignore[attr-defined]
    )

    assert_raises(
        FrozenInstanceError,
        lambda: setattr(
            transaction,
            "amount",
            Decimal("999.00"),
        ),
    )

In [12]:
def run_all_tests() -> None:
    """Последовательно запускает все тесты программы."""

    tests = (
        test_normalize_money,
        test_account_creation,
        test_account_constructor_validation,
        test_deposit,
        test_invalid_deposit,
        test_successful_withdrawal,
        test_insufficient_funds,
        test_credit_account,
        test_event_actor_validation,
        test_holder_permissions,
        test_employee_and_system_permissions,
        test_blocked_account_withdrawal,
        test_history_is_protected,
    )

    for test in tests:
        test()
        print(f"✅ {test.__name__}")

    print(f"\nВсе тесты пройдены: {len(tests)}")


run_all_tests()

✅ test_normalize_money
✅ test_account_creation
✅ test_account_constructor_validation
✅ test_deposit
✅ test_invalid_deposit
✅ test_successful_withdrawal
✅ test_insufficient_funds
✅ test_credit_account
✅ test_event_actor_validation
✅ test_holder_permissions
✅ test_employee_and_system_permissions
✅ test_blocked_account_withdrawal
✅ test_history_is_protected

Все тесты пройдены: 13


In [14]:
from zoneinfo import ZoneInfo


MOSCOW_TIMEZONE = ZoneInfo("Europe/Moscow")


def format_moscow_datetime(value: datetime) -> str:
    """Форматирует время в часовом поясе Москвы."""

    if value.tzinfo is None or value.utcoffset() is None:
        raise ValueError(
            "Дата и время должны содержать информацию "
            "о часовом поясе"
        )

    moscow_datetime = value.astimezone(MOSCOW_TIMEZONE)
    offset = moscow_datetime.utcoffset()

    if offset is None:
        raise ValueError(
            "Не удалось определить смещение часового пояса"
        )

    total_minutes = int(offset.total_seconds() // 60)
    sign = "+" if total_minutes >= 0 else "-"
    absolute_minutes = abs(total_minutes)
    hours, minutes = divmod(absolute_minutes, 60)

    timezone_name = moscow_datetime.tzname() or "MSK"

    return (
        f"{moscow_datetime:%d.%m.%Y %H:%M:%S} "
        f"{timezone_name} "
        f"(UTC{sign}{hours:02d}:{minutes:02d}, "
        f"{MOSCOW_TIMEZONE.key})"
    )


def format_money(
    value: Decimal,
    currency: Currency,
) -> str:
    """Форматирует денежную сумму с указанием валюты."""

    formatted_value = f"{value:,.2f}".replace(",", " ")

    return f"{formatted_value} {currency.value}"


def format_transaction_history(account: Account) -> str:
    """Формирует текстовое представление истории транзакций."""

    history = account.get_history()

    header = (
        "ИСТОРИЯ ТРАНЗАКЦИЙ\n"
        f"Счёт: {account.get_account_id()}\n"
        f"Владелец: {account.holder}\n"
        f"Валюта: {account.get_currency().value}\n"
        f"{'=' * 72}"
    )

    if not history:
        return f"{header}\nИстория транзакций пуста."

    operation_names = {
        OperationType.DEPOSIT: "Пополнение",
        OperationType.WITHDRAW: "Снятие",
    }

    status_names = {
        OperationStatus.SUCCESS: "Успешно",
        OperationStatus.FAIL: "Отказ",
    }

    failure_reason_names = {
        FailureReason.INSUFFICIENT_FUNDS: (
            "Недостаточно средств"
        ),
        FailureReason.CREDIT_LIMIT_EXCEEDED: (
            "Превышен кредитный лимит"
        ),
        FailureReason.ACCOUNT_BLOCKED: (
            "Счёт заблокирован"
        ),
    }

    lines = [header]

    for number, transaction in enumerate(
        history,
        start=1,
    ):
        operation_name = operation_names[
            transaction.operation_type
        ]
        status_name = status_names[
            transaction.status
        ]

        amount = format_money(
            transaction.amount,
            transaction.currency,
        )
        balance = format_money(
            transaction.balance_after,
            transaction.currency,
        )
        created_at = format_moscow_datetime(
            transaction.created_at
        )

        lines.extend(
            (
                f"\nТранзакция № {number}",
                f"ID: {transaction.transaction_id}",
                f"Операция: {operation_name}",
                f"Сумма: {amount}",
                f"Статус: {status_name}",
                f"Баланс после операции: {balance}",
                f"Время: {created_at}",
            )
        )

        if transaction.failure_reason is not None:
            failure_reason = failure_reason_names[
                transaction.failure_reason
            ]
            lines.append(
                f"Причина отказа: {failure_reason}"
            )

        if transaction.credit_used is not None:
            credit_used = (
                "да"
                if transaction.credit_used
                else "нет"
            )
            lines.append(
                "Использованы кредитные средства: "
                f"{credit_used}"
            )

        lines.append("-" * 72)

    return "\n".join(lines)


def format_account_events(account: Account) -> str:
    """Формирует текстовое представление событий счёта."""

    events = account.get_events_history()

    header = (
        "ИСТОРИЯ СОБЫТИЙ СЧЁТА\n"
        f"Счёт: {account.get_account_id()}\n"
        f"Владелец: {account.holder}\n"
        f"{'=' * 72}"
    )

    if not events:
        return f"{header}\nИстория событий пуста."

    event_names = {
        AccountEventType.BLOCKED: (
            "Счёт заблокирован"
        ),
        AccountEventType.UNBLOCKED: (
            "Счёт разблокирован"
        ),
    }

    actor_names = {
        ActorType.ACCOUNT_HOLDER: (
            "Владелец счёта"
        ),
        ActorType.BANK_EMPLOYEE: (
            "Сотрудник банка"
        ),
        ActorType.SYSTEM: (
            "Банковская система"
        ),
    }

    block_reason_names = {
        BlockReason.USER_REQUEST: (
            "Запрос владельца"
        ),
        BlockReason.FRAUD_SUSPICION: (
            "Подозрение на мошенничество"
        ),
        BlockReason.BANK_DECISION: (
            "Решение банка"
        ),
        BlockReason.LEGAL_RESTRICTION: (
            "Юридическое ограничение"
        ),
    }

    lines = [header]

    for number, event in enumerate(
        events,
        start=1,
    ):
        event_name = event_names[event.event_type]
        actor_name = actor_names[event.actor.actor_type]
        created_at = format_moscow_datetime(
            event.created_at
        )

        lines.extend(
            (
                f"\nСобытие № {number}",
                f"ID: {event.event_id}",
                f"Событие: {event_name}",
                f"Инициатор: {actor_name}",
                f"Время: {created_at}",
            )
        )

        if event.actor.actor_id is not None:
            lines.append(
                f"ID инициатора: {event.actor.actor_id}"
            )

        if event.block_reason is not None:
            block_reason = block_reason_names[
                event.block_reason
            ]
            lines.append(
                f"Причина блокировки: {block_reason}"
            )

        lines.append("-" * 72)

    return "\n".join(lines)


def format_account_journal(account: Account) -> str:
    """Формирует полный журнал транзакций и событий счёта."""

    return (
        f"{format_transaction_history(account)}\n\n"
        f"{format_account_events(account)}"
    )

In [16]:
def demonstrate_banking_program() -> None:
    """Демонстрирует основные возможности банковской программы."""

    print("=" * 72)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ БАНКОВСКОЙ ПРОГРАММЫ")
    print("=" * 72)

    # Создание участников
    holder_id = uuid4()

    owner = EventActor(
        actor_type=ActorType.ACCOUNT_HOLDER,
        actor_id=holder_id,
    )

    employee = EventActor(
        actor_type=ActorType.BANK_EMPLOYEE,
        actor_id=uuid4(),
    )

    system = EventActor(
        actor_type=ActorType.SYSTEM,
    )

    # Создание кредитного счёта
    account = CreditAccount(
        account_holder="Илья",
        balance=Decimal("1000.00"),
        credit_limit=Decimal("500.00"),
        currency=Currency.RUB,
        holder_id=holder_id,
    )

    print("\n1. Создан кредитный счёт")
    print(account)
    print(f"ID счёта: {account.get_account_id()}")
    print(f"ID владельца: {account.get_holder_id()}")
    print(
        "Статус счёта: "
        f"{account.get_account_status().value}"
    )

    # Пополнение
    deposit_transaction = account.deposit(
        Decimal("250.50")
    )

    print("\n2. Выполнено пополнение")
    print(
        "ID транзакции: "
        f"{deposit_transaction.transaction_id}"
    )
    print(
        "Сумма: "
        f"{format_money(
            deposit_transaction.amount,
            deposit_transaction.currency,
        )}"
    )
    print(
        "Статус: "
        f"{deposit_transaction.status.value}"
    )
    print(
        "Баланс: "
        f"{format_money(
            account.get_balance(),
            account.get_currency(),
        )}"
    )
    print(
        "Время: "
        f"{format_moscow_datetime(
            deposit_transaction.created_at
        )}"
    )

    # Успешное снятие
    withdrawal_transaction = account.withdraw(
        Decimal("400.00")
    )

    print("\n3. Выполнено успешное снятие")
    print(
        "ID транзакции: "
        f"{withdrawal_transaction.transaction_id}"
    )
    print(
        "Сумма: "
        f"{format_money(
            withdrawal_transaction.amount,
            withdrawal_transaction.currency,
        )}"
    )
    print(
        "Статус: "
        f"{withdrawal_transaction.status.value}"
    )
    print(
        "Баланс: "
        f"{format_money(
            account.get_balance(),
            account.get_currency(),
        )}"
    )
    print(
        "Время: "
        f"{format_moscow_datetime(
            withdrawal_transaction.created_at
        )}"
    )

    # Снятие с использованием кредитных средств
    credit_transaction = account.withdraw(
        Decimal("1000.00")
    )

    print("\n4. Выполнено снятие кредитных средств")
    print(
        "ID транзакции: "
        f"{credit_transaction.transaction_id}"
    )
    print(
        "Сумма: "
        f"{format_money(
            credit_transaction.amount,
            credit_transaction.currency,
        )}"
    )
    print(
        "Статус: "
        f"{credit_transaction.status.value}"
    )
    print(
        "Использованы кредитные средства: "
        f"{'да' if credit_transaction.credit_used else 'нет'}"
    )
    print(
        "Баланс: "
        f"{format_money(
            account.get_balance(),
            account.get_currency(),
        )}"
    )
    print(
        "Доступно средств: "
        f"{format_money(
            account.get_available_funds(),
            account.get_currency(),
        )}"
    )

    # Неуспешное снятие из-за превышения лимита
    failed_credit_transaction = account.withdraw(
        Decimal("400.00")
    )

    print("\n5. Попытка превысить кредитный лимит")
    print(
        "Статус: "
        f"{failed_credit_transaction.status.value}"
    )

    if failed_credit_transaction.failure_reason is not None:
        print(
            "Причина отказа: "
            f"{failed_credit_transaction.failure_reason.value}"
        )

    print(
        "Баланс не изменился: "
        f"{format_money(
            account.get_balance(),
            account.get_currency(),
        )}"
    )

    # Пополнение для восстановления положительного баланса
    repayment_transaction = account.deposit(
        Decimal("500.00")
    )

    print("\n6. Выполнено погашение задолженности")
    print(
        "Сумма: "
        f"{format_money(
            repayment_transaction.amount,
            repayment_transaction.currency,
        )}"
    )
    print(
        "Баланс: "
        f"{format_money(
            account.get_balance(),
            account.get_currency(),
        )}"
    )

    # Блокировка владельцем
    blocked_event = account.block_account(
        block_reason=BlockReason.USER_REQUEST,
        actor=owner,
    )

    print("\n7. Счёт заблокирован по просьбе владельца")
    print(
        "Статус счёта: "
        f"{account.get_account_status().value}"
    )

    block_reason = account.get_block_reason()

    if block_reason is not None:
        print(f"Причина: {block_reason.value}")

    if blocked_event is not None:
        print(f"ID события: {blocked_event.event_id}")
        print(
            "Время: "
            f"{format_moscow_datetime(
                blocked_event.created_at
            )}"
        )

    # Попытка снятия с заблокированного счёта
    blocked_withdrawal = account.withdraw(
        Decimal("50.00")
    )

    print("\n8. Попытка снятия с заблокированного счёта")
    print(f"Статус: {blocked_withdrawal.status.value}")

    if blocked_withdrawal.failure_reason is not None:
        print(
            "Причина отказа: "
            f"{blocked_withdrawal.failure_reason.value}"
        )

    print(
        "Баланс не изменился: "
        f"{format_money(
            account.get_balance(),
            account.get_currency(),
        )}"
    )

    # Владелец разблокирует счёт
    unblocked_event = account.unblock_account(owner)

    print("\n9. Счёт разблокирован владельцем")
    print(
        "Статус счёта: "
        f"{account.get_account_status().value}"
    )
    print(
        "Причина блокировки: "
        f"{account.get_block_reason()}"
    )

    if unblocked_event is not None:
        print(f"ID события: {unblocked_event.event_id}")
        print(
            "Время: "
            f"{format_moscow_datetime(
                unblocked_event.created_at
            )}"
        )

    # Системная блокировка
    system_blocked_event = account.block_account(
        block_reason=BlockReason.FRAUD_SUSPICION,
        actor=system,
    )

    print("\n10. Счёт заблокирован системой")
    print(
        "Статус счёта: "
        f"{account.get_account_status().value}"
    )

    if system_blocked_event is not None:
        print(
            "Инициатор: "
            f"{system_blocked_event.actor.actor_type.value}"
        )
        print(
            "Причина: "
            f"{system_blocked_event.block_reason.value}"
        )

    # Попытка системы снять установленную блокировку
    print("\n11. Система пытается разблокировать счёт")

    try:
        account.unblock_account(system)
    except PermissionError as error:
        print(f"Отказ в доступе: {error}")

    print(
        "Счёт остаётся заблокированным: "
        f"{account.is_blocked}"
    )

    # Сотрудник банка снимает системную блокировку
    employee_unblocked_event = account.unblock_account(
        employee
    )

    print("\n12. Сотрудник банка разблокировал счёт")
    print(
        "Статус счёта: "
        f"{account.get_account_status().value}"
    )

    if employee_unblocked_event is not None:
        print(
            "Инициатор: "
            f"{employee_unblocked_event.actor.actor_type.value}"
        )
        print(
            "Время: "
            f"{format_moscow_datetime(
                employee_unblocked_event.created_at
            )}"
        )

    # Итоговая информация
    print("\n" + "=" * 72)
    print("ИТОГОВОЕ СОСТОЯНИЕ СЧЁТА")
    print("=" * 72)
    print(account)
    print(
        "Количество транзакций: "
        f"{len(account.get_history())}"
    )
    print(
        "Количество событий: "
        f"{len(account.get_events_history())}"
    )

    # Полный журнал
    print("\n")
    print(format_account_journal(account))


demonstrate_banking_program()

ДЕМОНСТРАЦИЯ РАБОТЫ БАНКОВСКОЙ ПРОГРАММЫ

1. Создан кредитный счёт
Владелец: Илья. Баланс: 1000.00 RUB. Кредитный лимит: 500.00 RUB. Доступно средств: 1500.00 RUB.
ID счёта: a4aa7f3d-60ae-46eb-987c-4440f3b893f7
ID владельца: 9dcd9063-3844-4632-a0f1-8546249cdfb4
Статус счёта: active

2. Выполнено пополнение
ID транзакции: a94eaff1-3feb-4978-ad52-22d6a13a79cc
Сумма: 250.50 RUB
Статус: success
Баланс: 1 250.50 RUB
Время: 10.10.2026 13:49:27 MSK (UTC+03:00, Europe/Moscow)

3. Выполнено успешное снятие
ID транзакции: b1ec01a8-9a80-4fdc-a69a-af28e1f684c0
Сумма: 400.00 RUB
Статус: success
Баланс: 850.50 RUB
Время: 10.10.2026 13:49:27 MSK (UTC+03:00, Europe/Moscow)

4. Выполнено снятие кредитных средств
ID транзакции: 185ccbbb-c02b-4b84-8ec9-8b3d34bc3c36
Сумма: 1 000.00 RUB
Статус: success
Использованы кредитные средства: да
Баланс: -149.50 RUB
Доступно средств: 350.50 RUB

5. Попытка превысить кредитный лимит
Статус: fail
Причина отказа: credit_limit_exceeded
Баланс не изменился: -149.50 RUB
